# Model 4 - Original EfficientNetB2

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import yaml

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs" / "default.yaml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Khong tim thay thu muc new_project")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import build_tf_dataset
from src.preprocessing import CLASS_INDEX_TO_NAME, load_split_manifest
from src.reproducibility import set_global_seed

with open(PROJECT_ROOT / "configs" / "default.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

set_global_seed(config["seed"])

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "split_manifest.csv"
MODEL_DIR = PROJECT_ROOT / "models" / "model4_original_efficientnetb2"
REPORT_DIR = PROJECT_ROOT / "reports" / "model4_original_efficientnetb2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 240
BATCH_SIZE = config["batch_size"]
NUM_CLASSES = config["num_classes"]
CLASS_NAMES = [CLASS_INDEX_TO_NAME[index] for index in range(NUM_CLASSES)]

manifest = load_split_manifest(MANIFEST_PATH)
display(pd.crosstab(manifest["split"], manifest["class_name"]))

train_ds = build_tf_dataset(manifest, RAW_DIR, split="train", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=True, seed=config["seed"])
val_ds = build_tf_dataset(manifest, RAW_DIR, split="validation", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
test_ds = build_tf_dataset(manifest, RAW_DIR, split="test", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = tf.keras.layers.Lambda(lambda image: image * 255.0, name="to_255")(inputs)
backbone = tf.keras.applications.EfficientNetB2(weights="imagenet", include_top=False, input_tensor=x)
x = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs, name="model4_original_efficientnetb2")

model.compile(loss="categorical_crossentropy", optimizer="Adam", metrics=["accuracy"])
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(MODEL_DIR / "best.keras", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.3, patience=5, min_delta=0.001, verbose=1),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)
model.save(MODEL_DIR / "last.keras")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

best_model = tf.keras.models.load_model(MODEL_DIR / "best.keras")
val_metrics = best_model.evaluate(val_ds, return_dict=True)
test_metrics = best_model.evaluate(test_ds, return_dict=True)
print(f"Validation accuracy: {val_metrics['accuracy']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")

history_df = pd.DataFrame(history.history)
history_df["epoch"] = np.arange(1, len(history_df) + 1)
history_df.to_csv(REPORT_DIR / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "learning_curves.png", dpi=160)
plt.show()

y_true, y_pred = [], []
for images, labels in test_ds:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(best_model.predict(images, verbose=0), axis=1))

report_df = pd.DataFrame(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)).transpose()
report_df.to_csv(REPORT_DIR / "classification_report.csv")
display(report_df)

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(REPORT_DIR / "confusion_matrix.csv")
display(cm_df)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Original EfficientNetB2 - Test Confusion Matrix")
plt.tight_layout()
plt.savefig(REPORT_DIR / "confusion_matrix.png", dpi=160)
plt.show()